<div style="background:linear-gradient(120deg,#00553A 0%,#00704A 55%,#00A86A 100%);padding:28px 32px;border-radius:10px;border-bottom:8px solid #F5C242;color:white;font-family:Calibri,Arial,sans-serif">
<div style="color:#F5C242;font-weight:bold;letter-spacing:3px;font-size:12px">STG17 TECHNICAL WORKSHOP · DAY 2 · LABORATORY · 15:45–16:45</div>
<div style="font-size:38px;font-weight:bold;margin-top:6px;color:white">One Model, Many Jobs</div>
<div style="font-size:19px;font-style:italic;color:#E6F6EE">An LLM toolkit for statisticians — seven stations, two deliverables, zero licence cost</div>
<div style="font-size:12px;margin-top:14px;color:#E6F6EE">Emerging Issues, Emerging Practice · Innovating the Data Value Chain<br>African Development Bank · African Union (STATAFRIC) · National Institute of Statistics of Rwanda</div>
</div>

# One Model, Many Jobs — participant notebook

**What this notebook is.** A hands-on companion to the session slides. It contains seven *stations*, each a self-contained mini-lab that uses **only free tools**: the Gemini API free tier (Google AI Studio), the Groq free tier, and Google Colab. You choose **two stations**, spend about **20 minutes** on each, and leave with **one deliverable per station** saved in the `outputs/` folder.

**Nothing blocks you.** If a key is missing, a quota is reached or the network drops, the notebook switches to **DEMO mode** and returns pre-recorded example answers, so you can still follow the logic and complete the verification steps.

> 🔒 **Data rule for today:** everything in this notebook is **synthetic** (the fictional *Republic of Kivuland*) or **public**. Never paste microdata, personal identifiers, embargoed figures or credentials into a free AI tool — free-tier inputs may be used by providers to improve their products.

## Map of the notebook

| # | Section | Tool | Deliverable saved in `outputs/` | Best for |
|---|---|---|---|---|
| 0 | **Setup** (everyone, 5 min) | Colab | — | everyone |
| A | 🧑‍💻 **Code** — write, translate, debug | Groq | `A_function.py`, `A_debug_log.md` | IT & data processing |
| B | 📝 **Reports & methodological notes** | Gemini | `B_commentary.md`, `B_method_note.md` | methodology & analysis |
| C | 🖥️ **Presentations** | Gemini + python-pptx | `C_deck.pptx` | dissemination, communication |
| D | 📊 **Charts & graphics** | Gemini + matplotlib | `D_chart.png`, `D_chart_card.md` | methodology & analysis |
| E | 🎨 **Visual identity & logos** | Groq/Gemini + SVG | `E_logo_*.svg`, `E_palette.png` | communication |
| F | 🎧 **Documents & audio briefings** | NotebookLM + Gemini + gTTS | `F_grounded_answer.md`, `F_briefing.mp3` | dissemination |
| G | 👁️ **Multimodal with Gemini** | Gemini vision | `G_extraction_report.md` | IT & data processing |
| ✔ | **Wrap-up** (everyone, 5 min) | — | `prompt_log.csv`, `outputs.zip` | everyone |

**Suggested pairs by office profile:** IT & data processing → **A + G** · Methodology & analysis → **B + D** · Dissemination → **C + F** · Communication → **E + C**.

### How each station is organised
Every station follows the same pedagogical rhythm:

1. 🎯 **Goal & deliverable** — what you will produce.
2. 💡 **Concept** — the idea behind the technique, in two minutes.
3. ▶️ **Run** — guided cells, built with the **R·C·T·F·C** recipe (Role · Context · Task · Format · Check).
4. ✅ **Verify** — an automatic or manual check. *No output leaves a station unchecked.*
5. ✍️ **Your turn** — adapt the cell to your own office.
6. 🧠 **Check yourself** — a short question with a hidden answer.

---
<div style="background:#00704A;color:white;padding:12px 18px;border-radius:8px;border-left:10px solid #F5C242"><b style="font-size:20px">0 · Setup</b> &nbsp;·&nbsp; ⏱ 5 minutes &nbsp;·&nbsp; everyone</div>

### 0.1 Get two free API keys (no credit card)

| Provider | Where | Steps | Secret name in Colab |
|---|---|---|---|
| **Google Gemini** | [aistudio.google.com](https://aistudio.google.com) | Sign in → **Get API key** → Create | `GEMINI_API_KEY` |
| **Groq** | [console.groq.com](https://console.groq.com) | Sign in → **API Keys** → Create | `GROQ_API_KEY` |

**Store them safely.** In Colab, click the **🔑 key icon** in the left sidebar → *Add new secret* → paste the name and the value → switch **Notebook access** on. Never paste a key directly into a cell: notebooks get shared, and keys leak.

> ⚠️ **Rate limits (indicative, at the time of writing):** Gemini free tier ≈ 10–15 requests/minute on Flash models; Groq free tier ≈ 30 requests/minute per organisation. One person per key. If you see *429 / RESOURCE_EXHAUSTED*, wait a minute — the helpers retry automatically.

### 0.2 Run the setup cells below (in order)

In [ ]:
# @title 0.2 · Install libraries (≈ 30 seconds)
import sys, subprocess
pkgs = ["google-genai", "openai", "python-pptx", "gTTS", "tabulate"]
res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
print("✅ Libraries ready:" if res.returncode == 0 else "⚠️ pip reported a problem (fine if the libraries are already installed):", ", ".join(pkgs))

In [ ]:
# @title 0.3 · Load keys, connect to providers, define helpers
import os, re, json, time, textwrap, datetime, io, math, random
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML, Image as IPImage, Audio

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

# ---- 1. Keys: Colab secrets → environment variables → none (DEMO) ----
def _get_secret(name):
    try:
        from google.colab import userdata  # only exists in Colab
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    return os.environ.get(name)

GEMINI_API_KEY = _get_secret("GEMINI_API_KEY")
GROQ_API_KEY   = _get_secret("GROQ_API_KEY")

# ---- 2. Models (free tiers change — the helpers fall back automatically) ----
GEMINI_MODELS = ["gemini-2.5-flash", "gemini-flash-latest", "gemini-2.5-flash-lite", "gemini-2.0-flash"]
GROQ_MODELS   = ["llama-3.3-70b-versatile", "openai/gpt-oss-120b", "llama-3.1-8b-instant"]

gemini_client, groq_client = None, None
if GEMINI_API_KEY:
    try:
        from google import genai
        from google.genai import types as gtypes
        gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    except Exception as e:
        print("⚠️ Gemini client not created:", e)
if GROQ_API_KEY:
    try:
        from openai import OpenAI
        groq_client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")
    except Exception as e:
        print("⚠️ Groq client not created:", e)

# ---- 3. Prompt log (becomes prompt_log.csv — raw material for a prompt library) ----
PROMPT_LOG = []

def _log(station, task, provider, model, latency, prompt, answer, mode):
    PROMPT_LOG.append(dict(time=datetime.datetime.now().isoformat(timespec="seconds"), station=station,
                           task=task, provider=provider, model=model, mode=mode,
                           latency_s=round(latency, 2), prompt_chars=len(prompt), answer_chars=len(answer or ""),
                           prompt=prompt[:2000]))

def _is_rate_limit(e):  return any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED", "rate limit", "Rate limit"))
def _is_not_found(e):   return any(k in str(e) for k in ("404", "not found", "NOT_FOUND", "does not exist", "decommissioned"))

def _call_gemini(prompt, system, temperature, json_mode, images):
    parts = []
    for img in images or []:
        parts.append(gtypes.Part.from_bytes(data=img, mime_type="image/png"))
    parts.append(prompt)
    cfg = dict(temperature=temperature)
    if system: cfg["system_instruction"] = system
    if json_mode: cfg["response_mime_type"] = "application/json"
    models = list(GEMINI_MODELS)
    tried = set()
    while models:
        m = models.pop(0)
        if m in tried: continue
        tried.add(m)
        for attempt in range(3):
            try:
                r = gemini_client.models.generate_content(model=m, contents=parts, config=gtypes.GenerateContentConfig(**cfg))
                return r.text, m
            except Exception as e:
                if _is_rate_limit(e) and attempt < 2:
                    wait = 15 * (attempt + 1); print(f"⏳ Gemini rate limit — waiting {wait}s…"); time.sleep(wait); continue
                if _is_not_found(e):
                    if not models:  # auto-discover a free Flash model
                        try:
                            found = [x.name.split("/")[-1] for x in gemini_client.models.list() if "flash" in x.name and "image" not in x.name and "tts" not in x.name]
                            models += [f for f in found if f not in tried][:3]
                        except Exception: pass
                    break
                raise
    raise RuntimeError("No Gemini model available")

def _call_groq(prompt, system, temperature, json_mode):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    for m in GROQ_MODELS:
        for attempt in range(3):
            try:
                kw = dict(model=m, messages=msgs, temperature=temperature)
                if json_mode: kw["response_format"] = {"type": "json_object"}
                r = groq_client.chat.completions.create(**kw)
                return r.choices[0].message.content, m
            except Exception as e:
                if _is_rate_limit(e) and attempt < 2:
                    wait = 10 * (attempt + 1); print(f"⏳ Groq rate limit — waiting {wait}s…"); time.sleep(wait); continue
                if _is_not_found(e): break
                raise
    raise RuntimeError("No Groq model available")

def ask(prompt, *, task, station, provider="gemini", system=None, temperature=0.3, json_mode=False, images=None, quiet=False):
    """Send a prompt to a free LLM. Falls back: chosen provider → other provider (text only) → DEMO answer."""
    order = [provider] + ([p for p in ("gemini", "groq") if p != provider] if not images else [])
    t0 = time.time()
    for p in order:
        try:
            if p == "gemini" and gemini_client:
                ans, m = _call_gemini(prompt, system, temperature, json_mode, images)
            elif p == "groq" and groq_client:
                ans, m = _call_groq(prompt, system, temperature, json_mode)
            else:
                continue
            _log(station, task, p, m, time.time() - t0, prompt, ans, "live")
            if not quiet: print(f"🟢 LIVE · {p} · {m} · {time.time()-t0:.1f}s")
            return ans
        except Exception as e:
            print(f"⚠️ {p} failed: {str(e)[:160]}")
    ans = DEMO.get(task, "[DEMO] No pre-recorded answer for this task.")
    _log(station, task, "demo", "-", 0, prompt, ans, "demo")
    if not quiet: print("🟡 DEMO MODE · pre-recorded example answer (add your keys to go live)")
    return ans

def extract_json(text):
    """Parse JSON even when the model wraps it in ``` fences or adds a sentence."""
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        m = re.search(r"(\{.*\}|\[.*\])", t, flags=re.S)
        if m: return json.loads(m.group(1))
        raise

def show(text):  display(Markdown(text))

def banner(kind, text):
    colors = {"ok": ("#E8F5EF", "#00A86A", "✅"), "warn": ("#FBF3DE", "#D49A00", "⚠️"), "bad": ("#FBEFED", "#B83B2E", "⛔"), "info": ("#F4F7F5", "#0E7C86", "💡")}
    bg, bd, ic = colors[kind]
    display(HTML(f'<div style="background:{bg};border:1px solid {bd};border-radius:6px;padding:8px 12px;margin:6px 0;color:#231F20">{ic} {text}</div>'))

status = lambda ok: "🟢 connected" if ok else "🟡 not set → DEMO answers"
display(Markdown(f"""
| Provider | Status | Models tried in order |
|---|---|---|
| Gemini (Google AI Studio) | {status(gemini_client)} | {', '.join(GEMINI_MODELS)} |
| Groq | {status(groq_client)} | {', '.join(GROQ_MODELS)} |
"""))

In [ ]:
# @title 0.3b · DEMO answers (used only when no key works — you can ignore this cell)
# Pre-recorded example outputs so every station runs offline. Some contain deliberate errors
# (one wrong figure in B, two misreads in G) so that the verification steps have something to catch.
import json
DEMO = json.loads(r'''{
 "A1_code": "```python\n# Assumption: weights must be positive; rows with missing value OR missing weight are excluded.\nimport pandas as pd\nimport numpy as np\n\ndef weighted_mean(df, value_col, weight_col, group_col=None):\n    \"\"\"Weighted mean of `value_col` using `weight_col`.\n\n    Rows where the value or the weight is missing are excluded before computing.\n    Returns a float if `group_col` is None, otherwise a pandas Series indexed by group.\n    \"\"\"\n    d = df.dropna(subset=[value_col, weight_col])\n    if (d[weight_col] < 0).any():\n        raise ValueError(\"Weights must be non-negative\")\n    if group_col is None:\n        return float((d[value_col] * d[weight_col]).sum() / d[weight_col].sum())\n    num = (d[value_col] * d[weight_col]).groupby(d[group_col]).sum()\n    den = d[weight_col].groupby(d[group_col]).sum()\n    return num / den\n\ndef test_overall():\n    df = pd.DataFrame({\"v\": [1.0, 3.0], \"w\": [1, 3]})\n    assert abs(weighted_mean(df, \"v\", \"w\") - 2.5) < 1e-9\n\ndef test_missing_excluded():\n    df = pd.DataFrame({\"v\": [1.0, np.nan, 3.0], \"w\": [1, 5, 1]})\n    assert abs(weighted_mean(df, \"v\", \"w\") - 2.0) < 1e-9\n\ndef test_by_group():\n    df = pd.DataFrame({\"g\": [\"a\", \"a\", \"b\"], \"v\": [2.0, 4.0, 5.0], \"w\": [1, 1, 2]})\n    out = weighted_mean(df, \"v\", \"w\", \"g\")\n    assert abs(out[\"a\"] - 3.0) < 1e-9 and abs(out[\"b\"] - 5.0) < 1e-9\n```",
 "A2_translate": "### Explanation\n1. `use \"kivu_regions.dta\", clear` — loads the regional file, replacing any data in memory.\n2. `drop if missing(...)` — removes regions where 2025 inflation **or** the population weight is missing.\n3. `gen change_pp = ...` — creates the change in inflation between 2024 and 2025, in percentage points.\n4. `collapse (mean) ... [aw=population_k]` — collapses the file to **one row** containing the population-weighted means (analytic weights).\n5. `format ... %4.1f` and `list` — display with one decimal.\n\n### Python translation\n```python\nd = kivu.dropna(subset=[\"cpi_infl_2025\", \"population_k\"]).copy()\nd[\"change_pp\"] = d[\"cpi_infl_2025\"] - d[\"cpi_infl_2024\"]\nw = d[\"population_k\"]\nresult = pd.DataFrame({\n    \"cpi_infl_2025\": [(d[\"cpi_infl_2025\"] * w).sum() / w.sum()],\n    \"change_pp\":     [(d[\"change_pp\"] * w).sum() / w.sum()],\n}).round(1)\nprint(result)\n```\n\n**Differences to watch:** Stata's `collapse` with `[aw=]` drops observations with a missing value *per variable*, whereas the `dropna` above drops the row for all variables — equivalent here only because `cpi_infl_2024` has no missing values. `aw` weights are rescaled internally, which does not change a weighted mean.",
 "A3_debug": "### Diagnosis\nPython's `and` needs a single True/False, but `kivu.region == \"North\"` is a whole **Series** of True/False values (one per row). pandas cannot decide whether the entire Series is \"true\", so it raises *\"The truth value of a Series is ambiguous\"*.\n\n### Fix\n```python\nsubset = kivu[(kivu.region == \"North\") & (kivu.cpi_infl_2025 > 5)]\nprint(subset)\n```\nUse the element-wise operator `&` (and `|` for \"or\", `~` for \"not\"), and wrap **each** condition in parentheses because `&` binds more tightly than `==` and `>`.\n\n### Prevention\nFor multi-condition filters, `kivu.query('region == \"North\" and cpi_infl_2025 > 5')` accepts plain `and`/`or` and is often easier to read.",
 "B_commentary": "### Context\nThe National Statistics Office of Kivuland publishes annual consumer price inflation for its eight regions, aggregated to the national level using population weights.\n\n### Key findings\nNational inflation eased from 6.7% in 2024 to 5.8% in 2025, a fall of 0.9 pp. Inflation declined in every region. The largest fall was recorded in the West (−1.5 pp, to 5.4%), followed by the Lakes region (−1.4 pp). The Highlands continued to record the highest rate, at 8.2%, while the Capital recorded the lowest, at 4.6%. In the East, inflation eased to 7.1%, remaining well above the national rate.\n\n### Caveats\nRegional rates for the Highlands and Lakes should be interpreted with caution because rural outlets are under-represented in the price collection. Figures are annual averages and do not reflect month-to-month movements.",
 "B_method_note": "### 1. Definition\nAnnual consumer price inflation by region measures the percentage change in the average level of consumer prices in each region of Kivuland between one calendar year and the previous one. The national rate summarises the eight regional rates for the country as a whole.\n\n### 2. Data source and coverage\nPrices are collected monthly in all eight regions, with approximately 4,200 price quotes per month. Coverage of rural outlets is incomplete in two regions (Highlands and Lakes). [TO BE COMPLETED BY METHODOLOGIST: outlet sampling frame and product coverage.]\n\n### 3. Method of computation\nA Laspeyres-type index is compiled, using base-period expenditure weights. The annual rate is calculated as the change in the annual average index relative to the annual average of the previous year.\n\n### 4. Aggregation and weighting\nExpenditure weights are derived from the household budget survey. Regional indices are aggregated into the national index using regional population, expressed in thousands, as weights.\n\n### 5. Limitations and quality\nRural outlets are under-represented in the Highlands and Lakes regions, which may bias regional estimates for these areas. Users should interpret differences involving these regions with caution. [TO BE COMPLETED BY METHODOLOGIST: quality indicators and imputation practices.]\n\n### 6. Release and revision policy\nResults are published annually in March. Revisions are made only to correct errors and are announced in advance, in line with the office's release calendar.",
 "C_outline": "{\"deck_title\": \"Regional Inflation 2025\", \"subtitle\": \"Briefing for the Ministry of Finance \\u2014 what the new release tells us\", \"slides\": [{\"kicker\": \"HEADLINE\", \"title\": \"Inflation eased nationally in 2025\", \"bullets\": [\"Price growth slowed compared with the previous year at national level\", \"Every region recorded a slower pace of price increases\", \"The release follows the pre-announced calendar and standard methodology\"], \"notes\": \"Open with the main message: inflation eased nationally and the slowdown was broad-based. Stress that this is the regular annual release, compiled with the established methodology, so it is directly comparable with the previous year's figures.\"}, {\"kicker\": \"REGIONAL PICTURE\", \"title\": \"Regional gaps persist, highest in the Highlands\", \"bullets\": [\"The Highlands remain the region with the highest inflation\", \"The Capital and the Coast remain at the lower end of the range\", \"Western and Lakes regions saw the strongest slowdown\"], \"notes\": \"Move from the national picture to the regions. The ranking of regions changed little: the Highlands still face the fastest price increases. Point to the data slide for the exact figures rather than quoting them from memory.\"}, {\"kicker\": \"QUALITY\", \"title\": \"Read Highlands and Lakes results with care\", \"bullets\": [\"Rural outlets are under-represented in two regions\", \"Differences involving these regions carry more uncertainty\", \"Improving rural coverage is part of the next collection plan\"], \"notes\": \"Be transparent about quality. The under-representation of rural outlets in the Highlands and Lakes is known and documented in the methodological note. It does not invalidate the results but calls for caution when comparing regions.\"}, {\"kicker\": \"INNOVATION\", \"title\": \"Connectivity gaps limit the web-scraping pilot\", \"bullets\": [\"An experimental pilot tests online prices as a complementary source\", \"Online prices mainly reflect well-connected urban markets\", \"[CONFIRM] Pilot results will be labelled as experimental statistics\"], \"notes\": \"Introduce the experimental price-scraping pilot. Explain that low connectivity in several regions means online prices are not representative of all households, so the pilot complements rather than replaces field collection. Confirm the labelling decision with the methodology unit.\"}, {\"kicker\": \"NEXT STEPS\", \"title\": \"What the office will do next\", \"bullets\": [\"Publish the full regional tables and methodological note\", \"Strengthen rural price collection in under-covered regions\", \"Report back on the pilot at the next quarterly briefing\"], \"notes\": \"Close with concrete next steps and invite questions. Offer a technical follow-up session with the methodology team for ministry analysts who need more detail on weights and coverage.\"}]}",
 "D_spec": "{\"chart_type\": \"scatter\", \"x\": \"internet_hh_pct\", \"y\": \"dl_speed_mbps\", \"label\": \"region\", \"title\": \"Regions with more connected households also enjoy faster mobile speeds\", \"subtitle\": \"Households with internet access (%) vs median mobile download speed (Mbps), 2026\", \"x_label\": \"Households with internet access (%)\", \"y_label\": \"Median download speed (Mbps)\", \"sort_by\": null}",
 "D_critique": "### Alt text\nScatter chart of Kivuland's eight regions. Regions where a larger share of households has internet access also have faster median mobile download speeds. The Capital leads on both measures and the Highlands is lowest on both, showing a clear positive relationship (r = 0.87), with the West faster than its connection rate would suggest.\n\n### Checklist review\n| Item | Verdict | Reason |\n|---|---|---|\n| Zero baseline | PASS | Both axes start at zero |\n| Finding-style title | PASS | The title states the relationship |\n| Sensible ordering | PASS | Not applicable to a scatter; points are labelled |\n| Source and units shown | PASS | Units in subtitle and axis labels; source in footer |\n| Not relying on colour alone | PASS | Single colour; meaning carried by position and labels |\n\n**Suggestion:** explain the bubble size (population) in a legend or subtitle, not only in the footer.",
 "E_logos": "{\"concepts\": [{\"name\": \"Rising Nodes\", \"rationale\": \"Connected data points climbing like a growth curve \\u2014 networks, progress and openness.\", \"svg\": \"<svg xmlns=\\\"http://www.w3.org/2000/svg\\\" viewBox=\\\"0 0 200 200\\\" width=\\\"200\\\" height=\\\"200\\\"><circle cx=\\\"100\\\" cy=\\\"100\\\" r=\\\"92\\\" fill=\\\"#E8F5EF\\\"/><polyline points=\\\"40,140 75,110 105,122 150,62\\\" fill=\\\"none\\\" stroke=\\\"#00704A\\\" stroke-width=\\\"8\\\" stroke-linecap=\\\"round\\\" stroke-linejoin=\\\"round\\\"/><circle cx=\\\"40\\\" cy=\\\"140\\\" r=\\\"11\\\" fill=\\\"#00A86A\\\"/><circle cx=\\\"75\\\" cy=\\\"110\\\" r=\\\"11\\\" fill=\\\"#00A86A\\\"/><circle cx=\\\"105\\\" cy=\\\"122\\\" r=\\\"11\\\" fill=\\\"#00A86A\\\"/><circle cx=\\\"150\\\" cy=\\\"62\\\" r=\\\"15\\\" fill=\\\"#F5C242\\\"/><text x=\\\"100\\\" y=\\\"178\\\" font-family=\\\"Arial\\\" font-size=\\\"20\\\" font-weight=\\\"bold\\\" fill=\\\"#00553A\\\" text-anchor=\\\"middle\\\">KDIL</text></svg>\"}, {\"name\": \"Data Baobab\", \"rationale\": \"A stylised baobab whose branches end in data points \\u2014 rooted, enduring, distinctly African.\", \"svg\": \"<svg xmlns=\\\"http://www.w3.org/2000/svg\\\" viewBox=\\\"0 0 200 200\\\" width=\\\"200\\\" height=\\\"200\\\"><rect x=\\\"88\\\" y=\\\"95\\\" width=\\\"24\\\" height=\\\"60\\\" rx=\\\"8\\\" fill=\\\"#C4621D\\\"/><path d=\\\"M100 100 L60 60 M100 100 L100 45 M100 100 L140 60 M60 60 L45 40 M140 60 L155 40\\\" stroke=\\\"#C4621D\\\" stroke-width=\\\"7\\\" stroke-linecap=\\\"round\\\"/><circle cx=\\\"45\\\" cy=\\\"40\\\" r=\\\"10\\\" fill=\\\"#00A86A\\\"/><circle cx=\\\"60\\\" cy=\\\"60\\\" r=\\\"8\\\" fill=\\\"#00A86A\\\"/><circle cx=\\\"100\\\" cy=\\\"45\\\" r=\\\"12\\\" fill=\\\"#F5C242\\\"/><circle cx=\\\"140\\\" cy=\\\"60\\\" r=\\\"8\\\" fill=\\\"#00A86A\\\"/><circle cx=\\\"155\\\" cy=\\\"40\\\" r=\\\"10\\\" fill=\\\"#00A86A\\\"/><rect x=\\\"40\\\" y=\\\"155\\\" width=\\\"120\\\" height=\\\"6\\\" rx=\\\"3\\\" fill=\\\"#00704A\\\"/><text x=\\\"100\\\" y=\\\"188\\\" font-family=\\\"Arial\\\" font-size=\\\"20\\\" font-weight=\\\"bold\\\" fill=\\\"#231F20\\\" text-anchor=\\\"middle\\\">KDIL</text></svg>\"}, {\"name\": \"Signal Grid\", \"rationale\": \"A pixel grid forming a signal-strength icon \\u2014 connectivity measured cell by cell.\", \"svg\": \"<svg xmlns=\\\"http://www.w3.org/2000/svg\\\" viewBox=\\\"0 0 200 200\\\" width=\\\"200\\\" height=\\\"200\\\"><rect x=\\\"10\\\" y=\\\"10\\\" width=\\\"180\\\" height=\\\"180\\\" rx=\\\"28\\\" fill=\\\"#00553A\\\"/><rect x=\\\"45\\\" y=\\\"115\\\" width=\\\"22\\\" height=\\\"30\\\" rx=\\\"4\\\" fill=\\\"#9ED9C0\\\"/><rect x=\\\"75\\\" y=\\\"95\\\" width=\\\"22\\\" height=\\\"50\\\" rx=\\\"4\\\" fill=\\\"#57BD92\\\"/><rect x=\\\"105\\\" y=\\\"75\\\" width=\\\"22\\\" height=\\\"70\\\" rx=\\\"4\\\" fill=\\\"#10A06A\\\"/><rect x=\\\"135\\\" y=\\\"50\\\" width=\\\"22\\\" height=\\\"95\\\" rx=\\\"4\\\" fill=\\\"#F5C242\\\"/><text x=\\\"100\\\" y=\\\"175\\\" font-family=\\\"Arial\\\" font-size=\\\"20\\\" font-weight=\\\"bold\\\" fill=\\\"#FFFFFF\\\" text-anchor=\\\"middle\\\">KDIL</text></svg>\"}]}",
 "E_palette": "{\"palette\": [{\"role\": \"primary\", \"name\": \"Savanna Green\", \"hex\": \"#00704A\"}, {\"role\": \"secondary\", \"name\": \"Lake Teal\", \"hex\": \"#0E7C86\"}, {\"role\": \"accent\", \"name\": \"Sunrise Gold\", \"hex\": \"#F5C242\"}, {\"role\": \"dark text\", \"name\": \"Basalt\", \"hex\": \"#231F20\"}, {\"role\": \"light background\", \"name\": \"Morning Mist\", \"hex\": \"#F4F7F5\"}]}",
 "F_answer": "Regional price indices are combined into the national index using regional population as weights [CPI-2]. Expenditure weights inside each index come from the latest household budget survey and are updated every five years [CPI-2]. Regional comparisons involving the Highlands and Lakes should be made with caution, because rural outlets are under-represented there [CPI-3]. The experimental connectivity statistics come from crowdsourced tests that over-represent urban smartphone users, so they describe tested connections rather than all households [NET-2]. Regions with few tests are flagged and should not be used to rank regions [NET-3]. Experimental statistics may also be revised without the usual notice period [REV-2].",
 "F_script": "{\"lines\": [{\"speaker\": \"Amina\", \"text\": \"Welcome. This short briefing was generated with AI for training purposes, using verified notes from our statistics office.\"}, {\"speaker\": \"Kofi\", \"text\": \"Today: how our regional figures are weighted, and what to keep in mind when comparing regions.\"}, {\"speaker\": \"Amina\", \"text\": \"Regional price indices are combined using regional population as weights, and spending weights come from the household budget survey.\"}, {\"speaker\": \"Kofi\", \"text\": \"One caution: rural shops are under-represented in the Highlands and Lakes, so compare those regions carefully.\"}, {\"speaker\": \"Amina\", \"text\": \"Our connectivity numbers are experimental. They come from volunteer speed tests, which lean towards urban smartphone users.\"}, {\"speaker\": \"Kofi\", \"text\": \"And where tests are few, the region is flagged. Please do not use flagged regions to rank performance.\"}, {\"speaker\": \"Amina\", \"text\": \"That's the briefing. Check the methodological notes for full details before quoting any result.\"}]}",
 "G_chart": "{\"values\": {\"Capital\": 22.1, \"Coast\": 19.8, \"West\": 17.3, \"North\": 15.9, \"South\": 12.4, \"East\": 10.7, \"Lakes\": 8.1, \"Highlands\": 7.1}}",
 "G_form": "{\"household_id\": \"KV-0417-22\", \"region\": \"Lakes\", \"household_size\": 6, \"head_age\": 48, \"water_source\": \"Protected well\", \"internet_at_home\": \"No\"}"
}''')
print(f'{len(DEMO)} demo answers loaded')

In [ ]:
# @title 0.4 · House style (AfDB-inspired palette) for charts and slides
AFDB = dict(green="#00A86A", deep="#00704A", forest="#00553A", gold="#F5C242", ochre="#D49A00",
            teal="#0E7C86", terra="#C4621D", brick="#B83B2E", ink="#231F20", slate="#5E6964",
            mist="#F4F7F5", mint="#E8F5EF", sage="#D5DED9", grey="#A9B5B0")
RAMP = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]

import logging
from matplotlib import font_manager
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
_installed = {f.name for f in font_manager.fontManager.ttflist}
FONT = [f for f in ("Calibri", "Carlito", "Liberation Sans", "Arial", "DejaVu Sans") if f in _installed] or ["sans-serif"]
plt.rcParams.update({
    "font.family": FONT, "font.size": 11,
    "axes.edgecolor": AFDB["sage"], "axes.labelcolor": AFDB["slate"], "axes.titleweight": "bold",
    "axes.titlesize": 14, "axes.titlecolor": AFDB["ink"], "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": AFDB["slate"], "ytick.color": AFDB["slate"], "axes.grid": False,
    "figure.facecolor": "white", "axes.prop_cycle": plt.cycler(color=[AFDB[c] for c in ("green", "deep", "teal", "ochre", "terra", "brick")]),
})
fig, ax = plt.subplots(figsize=(8, 0.6))
for i, (k, v) in enumerate(list(AFDB.items())[:10]):
    ax.add_patch(plt.Rectangle((i, 0), 0.95, 1, color=v)); ax.text(i + .47, -.35, k, ha="center", fontsize=8, color=AFDB["slate"])
ax.set_xlim(0, 10); ax.set_ylim(-.6, 1); ax.axis("off"); plt.show()

### 0.5 The shared dataset: the *Republic of Kivuland* (synthetic)

All stations reuse one small, **entirely fictional** regional table, so you can concentrate on the technique rather than on the data. It echoes the week's themes: prices, connectivity (Day 3, Ookla) and population weights (WorldPop).

In [ ]:
# @title 0.5 · Load the synthetic Kivuland dataset
kivu = pd.DataFrame({
    "region":          ["Capital", "North", "South", "East", "West", "Lakes", "Highlands", "Coast"],
    "population_k":    [2450, 1320, 1780, 960, 1150, 870, 640, 1430],
    "cpi_infl_2024":   [5.8, 7.2, 6.1, 8.4, 6.9, 7.7, 9.1, 5.5],
    "cpi_infl_2025":   [4.6, 6.8, 5.2, 7.9, 5.4, 6.3, 8.2, 4.9],
    "internet_hh_pct": [71.4, 38.2, 52.6, 29.5, 44.8, 33.1, 21.7, 58.3],
    "dl_speed_mbps":   [22.1, 15.9, 12.4, 10.7, 17.3, 8.1, 7.4, 19.8],
})
kivu["infl_change_pp"] = (kivu.cpi_infl_2025 - kivu.cpi_infl_2024).round(1)
DATA_NOTE = ("SYNTHETIC DATA — Republic of Kivuland (fictional). population_k = population in thousands; "
             "cpi_infl_* = annual CPI inflation, %; internet_hh_pct = households with internet access, %; "
             "dl_speed_mbps = median mobile download speed, Mbps; infl_change_pp = change 2024→2025, percentage points.")
print(DATA_NOTE)
kivu

> 🧠 **Check yourself.** Why does this notebook use a synthetic country instead of your own national data?
> <details><summary>Show answer</summary>
> Because free tiers are not covered by an institutional data-processing agreement, and providers may use inputs to improve their products. Synthetic or already-published data is "green" on the traffic light. It also lets every participant compare results on the same table.
> </details>

---
<div style="background:#00A86A;color:white;padding:14px 18px;border-radius:8px;border-left:10px solid #F5C242;font-family:Calibri,Arial,sans-serif"><span style="font-size:13px;letter-spacing:3px;color:#F5C242;font-weight:bold">STATION A</span><br><b style="font-size:24px">Code — write, translate and debug</b><br><span style="font-size:13px">⏱ 20 min &nbsp;·&nbsp; 🧰 Groq (fast open-weight models) &nbsp;·&nbsp; 🎯 Deliverable: <code>A_function.py</code> + <code>A_debug_log.md</code></span></div>

### 🎯 Goal
Use an LLM as a **pair programmer** for three everyday jobs in a statistical office: writing a new function *with tests*, translating legacy code (Stata → Python), and diagnosing an error.

### 💡 Concept — the model is a fast junior colleague, not an oracle
* LLM-generated code is often **plausible and wrong**: it runs, but computes the wrong thing (unweighted mean instead of weighted, wrong join key…).
* The antidote is to ask for **tests at the same time as the code**, and to compare the result with a **reference value you trust**.
* We use **Groq** here: its speed makes the *generate → test → fix* loop feel interactive (you measured this in the benchmark session).

| Rule | Why |
|---|---|
| Read before you run | Generated code can delete files, call the network, or silently drop rows |
| Ask for the diagnosis before the fix | You learn the cause; the fix is easier to review |
| Test against a known answer | "It runs" ≠ "it is right" |

### ▶️ A1 · Generate a function *with its tests*
The prompt below follows the **R·C·T·F·C** recipe. Read it before running.

In [ ]:
A1_PROMPT = f"""
ROLE: You are a senior data engineer in a national statistics office. You write clear, tested pandas code.
CONTEXT: A DataFrame has these columns: {list(kivu.columns)}.
TASK: Write a Python function `weighted_mean(df, value_col, weight_col, group_col=None)` that returns the
weighted mean of value_col using weight_col, either overall (float) or by group (pandas Series).
FORMAT: Return ONE Python code block only, containing:
  1. the function with a docstring,
  2. rows with a missing value or missing weight must be excluded (document this),
  3. three test functions named test_* using plain assert statements.
CHECK: If anything in the task is ambiguous, state your assumption in a comment at the top.
"""
a1_answer = ask(A1_PROMPT, task="A1_code", station="A", provider="groq", temperature=0.1)
a1_code = re.search(r"```(?:python)?\n(.*?)```", a1_answer, flags=re.S)
a1_code = a1_code.group(1) if a1_code else a1_answer
show(f"```python\n{a1_code}\n```")

### ✅ A1 · Verify — read, run in a sandbox, compare with a reference
Read the code above first. Then set `I_HAVE_READ_THE_CODE = True` and run the cell. The code runs in a separate namespace, its own tests are executed, and the result is compared with a reference computed by hand.

In [ ]:
I_HAVE_READ_THE_CODE = True   # ← set to False to practise the habit: nothing runs until you have read it

reference = (kivu.cpi_infl_2025 * kivu.population_k).sum() / kivu.population_k.sum()
if not I_HAVE_READ_THE_CODE:
    banner("warn", "Read the generated code first, then set I_HAVE_READ_THE_CODE = True.")
else:
    ns = {"pd": pd, "np": np}
    try:
        exec(a1_code, ns)
        tests = [k for k in ns if k.startswith("test_") and callable(ns[k])]
        results = []
        for t in tests:
            try: ns[t](); results.append((t, "✅ pass"))
            except Exception as e: results.append((t, f"❌ {type(e).__name__}: {e}"))
        display(pd.DataFrame(results, columns=["model-written test", "result"]))
        got = ns["weighted_mean"](kivu, "cpi_infl_2025", "population_k")
        ok = abs(float(got) - reference) < 1e-9
        banner("ok" if ok else "bad", f"National weighted inflation 2025 — generated function: <b>{float(got):.4f}</b> · reference: <b>{reference:.4f}</b> → {'match' if ok else 'MISMATCH — do not use this code'}")
        by = ns["weighted_mean"](kivu, "cpi_infl_2025", "population_k", "region")
        display(by.round(2).to_frame("weighted mean (by region)").T)
        (OUT / "A_function.py").write_text("# Generated with an LLM, reviewed and tested in the STG17 lab\n" + a1_code)
        print("💾 saved outputs/A_function.py")
    except Exception as e:
        banner("bad", f"The generated code failed: {type(e).__name__}: {e}. Paste this error into A3 below!")

> 🧠 **Check yourself.** The model's own tests pass. Is that enough to trust the function?
> <details><summary>Show answer</summary>
> No. Tests written by the same model can share its misunderstanding (for example, a test that also expects an unweighted mean). That is why we added an <b>independent reference value</b>. In production, the tests should come from the statistician who owns the method.
> </details>

### ▶️ A2 · Translate legacy code (Stata → Python) and explain it
Many offices hold years of Stata, SPSS or SAS scripts. LLMs are good at translation *and* at explaining what old code does — useful for documentation and handovers.

In [ ]:
STATA_SNIPPET = """
* Regional inflation, population-weighted, from the regional file
use "kivu_regions.dta", clear
drop if missing(cpi_infl_2025) | missing(population_k)
gen change_pp = cpi_infl_2025 - cpi_infl_2024
collapse (mean) cpi_infl_2025 change_pp [aw=population_k]
format cpi_infl_2025 change_pp %4.1f
list
"""
A2_PROMPT = f"""
ROLE: You are a statistician fluent in both Stata and Python/pandas.
CONTEXT: Legacy Stata script:
{STATA_SNIPPET}
TASK: (1) Explain in plain English, line by line, what the script does. (2) Translate it to pandas,
assuming a DataFrame called `kivu` is already loaded.
FORMAT: Markdown with two headings, "Explanation" and "Python translation" (one code block).
CHECK: Point out any behaviour that differs between Stata and pandas (e.g. how [aw=] weights and missing values work).
"""
a2_answer = ask(A2_PROMPT, task="A2_translate", station="A", provider="groq", temperature=0.1)
show(a2_answer)

### ▶️ A3 · Debug: ask for the *diagnosis* before the *fix*
Below is a bug that every pandas user meets once. We capture the real traceback and send it with the code.

In [ ]:
import traceback
BUGGY_CODE = """
subset = kivu[kivu.region == "North" and kivu.cpi_infl_2025 > 5]
print(subset)
"""
try:
    exec(BUGGY_CODE, {"kivu": kivu})
    trace = "no error"
except Exception:
    trace = traceback.format_exc(limit=2)
print(trace)

A3_PROMPT = f"""
ROLE: You are a patient Python mentor for statisticians.
CONTEXT: This code:
{BUGGY_CODE}
raised this error:
{trace}
TASK: First explain the ROOT CAUSE in two or three sentences a beginner understands.
Then give the corrected code, then one tip to avoid this class of error.
FORMAT: Markdown with headings "Diagnosis", "Fix", "Prevention".
CHECK: Do not change what the code is meant to select.
"""
a3_answer = ask(A3_PROMPT, task="A3_debug", station="A", provider="groq", temperature=0.1)
show(a3_answer)

In [ ]:
# ✅ Verify the fix yourself — run the corrected selection and inspect it
fixed = kivu[(kivu.region == "North") & (kivu.cpi_infl_2025 > 5)]
display(fixed)
(OUT / "A_debug_log.md").write_text(f"# Debugging log — Station A\n\n## Code\n```python\n{BUGGY_CODE}\n```\n\n## Error\n```\n{trace}\n```\n\n## LLM diagnosis\n{a3_answer}\n\n## Human verification\nCorrected selection returned {len(fixed)} row(s): {', '.join(fixed.region)}.\n")
print("💾 saved outputs/A_debug_log.md")

### ✍️ Your turn (A)
Paste a short script from **your own office** (no credentials, no confidential paths or data) into `MY_CODE`, choose a job, and run.

In [ ]:
MY_CODE = """
# paste a short R, Stata, SPSS, SAS or Python snippet here
"""
MY_JOB = "Explain this code line by line, then suggest three improvements for readability and robustness."
if MY_CODE.strip().startswith("# paste"):
    banner("info", "Paste your snippet in MY_CODE first.")
else:
    show(ask(f"ROLE: Senior statistical programmer.\nCONTEXT:\n{MY_CODE}\nTASK: {MY_JOB}\nFORMAT: Markdown.\nCHECK: Flag anything you are unsure about.",
             task="A_yourturn", station="A", provider="groq"))

---
<div style="background:#00704A;color:white;padding:14px 18px;border-radius:8px;border-left:10px solid #F5C242;font-family:Calibri,Arial,sans-serif"><span style="font-size:13px;letter-spacing:3px;color:#F5C242;font-weight:bold">STATION B</span><br><b style="font-size:24px">Reports & methodological notes</b><br><span style="font-size:13px">⏱ 20 min &nbsp;·&nbsp; 🧰 Gemini API (free tier) &nbsp;·&nbsp; 🎯 Deliverable: <code>B_commentary.md</code> (fact-checked) + <code>B_method_note.md</code></span></div>

### 🎯 Goal
Draft a short **statistical commentary** and a **methodological note** — and prove automatically that every figure in the text exists in the source table.

### 💡 Concept — *computers compute, language models narrate*

```
   pandas / R / Stata               LLM                        Python
 ┌────────────────────┐    ┌──────────────────────┐    ┌──────────────────────┐
 │ compute every      │ →  │ turn the verified    │ →  │ extract every number │
 │ figure from data   │    │ table into prose     │    │ and check it exists  │
 └────────────────────┘    └──────────────────────┘    └──────────────────────┘
```

A language model predicts plausible words; it does not *look up* your numbers. If you ask it for a report without data, it will invent figures that look right. So we (1) compute the figures ourselves, (2) give the model **only** that table, and (3) run a **fact-check**.

### ▶️ B1 · Compute the figures first (no AI involved)

In [ ]:
w = kivu.population_k
facts = {
    "national_infl_2024": round((kivu.cpi_infl_2024 * w).sum() / w.sum(), 1),
    "national_infl_2025": round((kivu.cpi_infl_2025 * w).sum() / w.sum(), 1),
}
facts["national_change_pp"] = round(facts["national_infl_2025"] - facts["national_infl_2024"], 1)
facts["highest_2025"] = kivu.loc[kivu.cpi_infl_2025.idxmax(), ["region", "cpi_infl_2025"]].tolist()
facts["lowest_2025"]  = kivu.loc[kivu.cpi_infl_2025.idxmin(), ["region", "cpi_infl_2025"]].tolist()
facts["largest_fall"] = kivu.loc[kivu.infl_change_pp.idxmin(), ["region", "infl_change_pp"]].tolist()
source_table = kivu[["region", "population_k", "cpi_infl_2024", "cpi_infl_2025", "infl_change_pp"]]
facts_md = "\n".join(f"- {k}: {v}" for k, v in facts.items())
show("**Verified facts (computed in pandas):**\n\n" + facts_md)

### ▶️ B2 · Draft the commentary from the table *only*

In [ ]:
B_SYSTEM = "You are a senior price statistician at the National Statistics Office of Kivuland. You write neutral, precise, plain-English commentary for ministry officials. You never use figures that are not supplied."
B_PROMPT = f"""
CONTEXT — the ONLY data you may use:
{source_table.to_markdown(index=False)}

Pre-computed national facts (population-weighted):
{facts_md}

TASK: Write a commentary of about 200 words on regional consumer price inflation in 2025 compared with 2024.
FORMAT: Markdown with exactly three level-3 headings: "Context", "Key findings", "Caveats".
Percentages with one decimal. Changes in percentage points ("pp").
CHECK: If you need a figure that is not in the data above, write [MISSING] instead of estimating it.
"""
b_draft = ask(B_PROMPT, system=B_SYSTEM, task="B_commentary", station="B", temperature=0.2)
show(b_draft)

### ✅ B3 · Automatic fact-check
The function below extracts every number with a decimal point (or followed by `%` / `pp`) and checks whether it appears in the source table or in the pre-computed facts. Years are ignored.

*In DEMO mode, the pre-recorded draft deliberately contains **one wrong figure** — can the check find it?*

In [ ]:
def fact_check(text, table, facts):
    allowed = set()
    for v in table.select_dtypes("number").to_numpy().ravel():
        allowed |= {round(float(v), 1), round(abs(float(v)), 1)}
    for v in facts.values():
        for x in (v if isinstance(v, list) else [v]):
            if isinstance(x, (int, float)): allowed |= {round(float(x), 1), round(abs(float(x)), 1)}
    found = re.finditer(r"(?<![\w.,])([-−+]?\d+(?:\.\d+)?)(\s*(?:%|pp\b|percentage point|per cent))?", text)
    rows = []
    for m in found:
        raw = m.group(1).replace("−", "-")
        if re.fullmatch(r"[-+]?(19|20)\d\d", raw): continue      # years
        if "." not in raw and not m.group(2): continue           # plain integers (headings, counts)
        val = round(float(raw), 1)
        ctx = text[max(0, m.start() - 35): m.end() + 10].replace("\n", " ")
        rows.append({"number": raw, "in source?": "✅" if abs(val) in allowed or val in allowed else "❌ NOT FOUND", "context": f"…{ctx}…"})
    return pd.DataFrame(rows)

fc = fact_check(b_draft, source_table, facts)
display(fc)
bad = fc[fc["in source?"] != "✅"] if len(fc) else fc
if len(bad):
    banner("bad", f"{len(bad)} figure(s) not traceable to the source. Correct them before release: {', '.join(bad.number)}")
else:
    banner("ok", "Every figure in the draft is traceable to the source table.")

In [ ]:
# Save the deliverable WITH its verification report
(OUT / "B_commentary.md").write_text(
    f"# Regional inflation commentary — DRAFT (AI-assisted)\n\n{b_draft}\n\n---\n## Fact-check report\n\n"
    + (fc.to_markdown(index=False) if len(fc) else "No figures found.")
    + "\n\n*Source: synthetic Kivuland dataset. Draft generated with an LLM; figures computed in pandas; reviewed by: ____________*\n")
print("💾 saved outputs/B_commentary.md")

### ▶️ B4 · Draft a methodological note with a fixed structure
Methodological notes follow a template. Giving the model **your headings** and **the facts** produces a first draft in seconds, which the methodologist then corrects. The headings below are a simplified version of the metadata structures used by many NSOs; replace them with your office's own template.

In [ ]:
METHOD_FACTS = """
- Indicator: annual consumer price inflation by region, Kivuland (synthetic example)
- Source: monthly price collection in 8 regions, ~ 4,200 price quotes per month
- Index formula: Laspeyres-type, base period weights from the household budget survey
- Regional aggregation to national: weighted by regional population (thousands)
- Annual rate: change in the annual average index versus the previous year
- Publication: annually in March; revisions only for errors, announced in advance
- Known limitation: rural outlets under-represented in two regions (Highlands, Lakes)
"""
HEADINGS = ["1. Definition", "2. Data source and coverage", "3. Method of computation",
            "4. Aggregation and weighting", "5. Limitations and quality", "6. Release and revision policy"]
B4_PROMPT = f"""
ROLE: Methodologist writing official metadata.
CONTEXT (use only these facts): {METHOD_FACTS}
TASK: Draft a methodological note.
FORMAT: Markdown, using exactly these level-3 headings in this order: {HEADINGS}. 40–70 words per section. Formal, neutral tone.
CHECK: If a section cannot be completed from the facts, write "[TO BE COMPLETED BY METHODOLOGIST]" rather than inventing.
"""
b_note = ask(B4_PROMPT, task="B_method_note", station="B", temperature=0.2)
show(b_note)
missing_sections = [h for h in HEADINGS if h.split(". ", 1)[1].lower() not in b_note.lower()]
banner("ok" if not missing_sections else "warn", "All required headings present." if not missing_sections else f"Missing headings: {missing_sections}")
(OUT / "B_method_note.md").write_text("# Methodological note — DRAFT (AI-assisted)\n\n" + b_note)
print("💾 saved outputs/B_method_note.md")

### ✍️ Your turn (B)
Change `B_PROMPT` so the commentary targets **journalists** (shorter sentences, one headline, no jargon) — then rerun B2 and B3. Does the fact-check still pass?

> 🧠 **Check yourself.** The fact-check passes. Does that make the commentary correct?
> <details><summary>Show answer</summary>
> Not necessarily. The check proves each number <i>exists</i> in the source, not that it is attached to the <i>right region or year</i> ("North fell to 4.6%" would pass because 4.6 exists — for the Capital). A human reviewer still reads every sentence against the table.
> </details>

---
<div style="background:#0E7C86;color:white;padding:14px 18px;border-radius:8px;border-left:10px solid #F5C242;font-family:Calibri,Arial,sans-serif"><span style="font-size:13px;letter-spacing:3px;color:#F5C242;font-weight:bold">STATION C</span><br><b style="font-size:24px">Presentations — from brief to .pptx</b><br><span style="font-size:13px">⏱ 20 min &nbsp;·&nbsp; 🧰 Gemini API + python-pptx &nbsp;·&nbsp; 🎯 Deliverable: <code>C_deck.pptx</code> in house colours</span></div>

### 🎯 Goal
Turn a short brief into a **structured slide outline** with an LLM, then render it as a **PowerPoint file in house style** with Python — including a native, editable chart built from verified data.

### 💡 Concept — separate *content* from *design*
Asking a chatbot to "make slides" gives inconsistent looks. The professional pattern is:

| Step | Who does it | Output |
|---|---|---|
| 1. Structure the story | **LLM** | JSON outline: titles, bullets, speaker notes |
| 2. Validate the structure | **Python** | reject missing fields, too many bullets |
| 3. Apply the design | **python-pptx** | every deck looks institutional |
| 4. Insert the data | **pandas → native chart** | numbers never pass through the LLM |

**Structured output (JSON)** is the key skill: it makes an LLM answer *machine-readable*, which is what lets you automate safely.

### ▶️ C1 · Brief → JSON outline

In [ ]:
BRIEF = """Audience: senior officials of the Ministry of Finance of Kivuland (synthetic example).
Purpose: 10-minute briefing on the 2025 regional inflation release.
Key messages: inflation eased nationally; regional gaps persist, highest in the Highlands;
connectivity gaps are a caveat for the new experimental price-scraping pilot.
Tone: neutral, factual. Do NOT quote any figures — the data slide is added automatically."""

C_PROMPT = f"""
ROLE: Communication officer at a national statistics office.
CONTEXT: {BRIEF}
TASK: Design a 5-slide briefing (the title slide and the data slide are added separately).
FORMAT: Return JSON only, with this schema:
{{"deck_title": str, "subtitle": str,
  "slides": [{{"kicker": str (2-3 words, uppercase), "title": str (max 70 chars, states the message),
              "bullets": [str, ...] (3 or 4 items, max 110 chars each), "notes": str (speaker notes, 40-80 words)}}]}}
CHECK: No numbers in bullets or titles. If a key message is unclear, add a bullet starting with "[CONFIRM]".
"""
c_raw = ask(C_PROMPT, task="C_outline", station="C", json_mode=True, temperature=0.4)
outline = extract_json(c_raw)
print(json.dumps(outline, indent=2, ensure_ascii=False)[:1500], "…")

### ✅ C2 · Validate the outline before rendering

In [ ]:
def validate_outline(o):
    problems = []
    for k in ("deck_title", "subtitle", "slides"):
        if k not in o: problems.append(f"missing key '{k}'")
    for i, s in enumerate(o.get("slides", []), 1):
        for k in ("kicker", "title", "bullets", "notes"):
            if k not in s: problems.append(f"slide {i}: missing '{k}'")
        if not 3 <= len(s.get("bullets", [])) <= 4: problems.append(f"slide {i}: {len(s.get('bullets', []))} bullets (expected 3–4)")
        if len(s.get("title", "")) > 80: problems.append(f"slide {i}: title too long")
        for b in s.get("bullets", []):
            if re.search(r"\d+\.\d|\d+\s*%", b): problems.append(f"slide {i}: figure found in bullet → '{b[:40]}…'")
    return problems

problems = validate_outline(outline)
if problems:
    banner("warn", "Outline issues — fix the prompt or the JSON before rendering:<br>" + "<br>".join(problems))
else:
    banner("ok", f"Outline valid: {len(outline['slides'])} slides, all fields present, no stray figures.")

### ▶️ C3 · Render the deck in house style (python-pptx)

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE, XL_LABEL_POSITION, XL_LEGEND_POSITION
from pptx.enum.text import MSO_ANCHOR

RGB = lambda h: RGBColor.from_string(h.lstrip("#"))

def _box(slide, x, y, w, h, text, size=14, bold=False, color="ink", italic=False, middle=False):
    tb = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = tb.text_frame; tf.word_wrap = True
    if middle: tf.vertical_anchor = MSO_ANCHOR.MIDDLE
    p = tf.paragraphs[0]; r = p.add_run(); r.text = text
    r.font.size = Pt(size); r.font.bold = bold; r.font.italic = italic; r.font.name = "Calibri"; r.font.color.rgb = RGB(AFDB[color])
    return tf

def _rect(slide, x, y, w, h, color, shape=MSO_SHAPE.RECTANGLE):
    s = slide.shapes.add_shape(shape, Inches(x), Inches(y), Inches(w), Inches(h))
    s.fill.solid(); s.fill.fore_color.rgb = RGB(AFDB[color]); s.line.fill.background(); return s

def _frame(prs, kicker, title, n):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    _rect(s, 0, 0, 13.333, 0.08, "green"); _rect(s, 0, 0, 1.8, 0.08, "gold")
    _box(s, 0.6, 0.35, 9, 0.3, kicker.upper(), 12, True, "deep")
    _box(s, 0.6, 0.65, 12, 0.9, title, 30, True)
    _box(s, 0.6, 7.0, 9, 0.3, outline["deck_title"] + " · AI-assisted draft", 10, color="slate")
    _box(s, 12.2, 7.0, 0.6, 0.3, str(n), 10, color="slate")
    return s

def render_deck(outline, df, path):
    prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
    # title slide
    s = prs.slides.add_slide(prs.slide_layouts[6])
    _rect(s, 0, 0, 13.333, 7.5, "forest"); _rect(s, 7.8, 0, 5.533, 7.5, "deep"); _rect(s, 0, 7.3, 13.333, 0.2, "gold")
    _box(s, 0.8, 1.6, 8, 0.4, "NATIONAL STATISTICS OFFICE OF KIVULAND · BRIEFING", 13, True, "gold")
    _box(s, 0.8, 2.2, 8.5, 2.2, outline["deck_title"], 44, True, "mist")
    _box(s, 0.8, 4.5, 8, 1, outline["subtitle"], 20, italic=True, color="mint")
    n = 2
    for sl in outline["slides"]:
        s = _frame(prs, sl["kicker"], sl["title"], n); n += 1
        for i, b in enumerate(sl["bullets"]):
            y = 1.9 + i * 1.2
            _rect(s, 0.6, y, 11.9, 1.0, "mist", MSO_SHAPE.ROUNDED_RECTANGLE)
            c = _rect(s, 0.85, y + 0.22, 0.56, 0.56, ["green", "deep", "teal", "ochre"][i], MSO_SHAPE.OVAL)
            c.text_frame.text = str(i + 1); c.text_frame.paragraphs[0].runs[0].font.bold = True
            c.text_frame.paragraphs[0].runs[0].font.size = Pt(16)
            _box(s, 1.7, y, 10.6, 1.0, b, 18, middle=True)
        s.notes_slide.notes_text_frame.text = sl["notes"]
    # data slide — numbers come from pandas, never from the LLM
    s = _frame(prs, "DATA", "Annual CPI inflation by region, 2024 and 2025 (%)", n)
    cd = CategoryChartData(); d = df.sort_values("cpi_infl_2025", ascending=False)
    cd.categories = list(d.region)
    cd.add_series("2024", list(d.cpi_infl_2024)); cd.add_series("2025", list(d.cpi_infl_2025))
    ch = s.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(0.6), Inches(1.7), Inches(12.1), Inches(5.1), cd).chart
    ch.has_legend = True; ch.legend.position = XL_LEGEND_POSITION.TOP; ch.legend.include_in_layout = False
    ch.value_axis.has_major_gridlines = False; ch.value_axis.tick_labels.font.size = Pt(11)
    ch.category_axis.tick_labels.font.size = Pt(12)
    for ser, col in zip(ch.series, ("grey", "green")):
        ser.format.fill.solid(); ser.format.fill.fore_color.rgb = RGB(AFDB[col])
        ser.data_labels.show_value = True; ser.data_labels.number_format = "0.0"; ser.data_labels.number_format_is_linked = False
        ser.data_labels.position = XL_LABEL_POSITION.OUTSIDE_END; ser.data_labels.font.size = Pt(10)
    _box(s, 0.6, 6.75, 11, 0.3, "Source: synthetic Kivuland dataset (training example). Figures computed in pandas.", 10, italic=True, color="slate")
    s.notes_slide.notes_text_frame.text = "This chart is generated directly from the verified table; no figure passed through the language model."
    prs.save(path)
    return n

if not problems:
    n_slides = render_deck(outline, kivu, OUT / "C_deck.pptx")
    banner("ok", f"💾 saved outputs/C_deck.pptx — {n_slides} slides (title + {len(outline['slides'])} content + 1 native data chart). Download it from the Files panel.")

### ✍️ Your turn (C)
Rewrite `BRIEF` for a real release of your office (public information only), rerun C1–C3, and open the `.pptx`. Replace the colours in `AFDB` with your office's palette to rebrand every future deck in one line.

> 🧠 **Check yourself.** Why does the prompt forbid figures in bullets, and how is that enforced?
> <details><summary>Show answer</summary>
> Figures in slides must come from verified data. The prompt asks the model not to write numbers, and <code>validate_outline()</code> enforces it with a regular expression. The only numbers in the deck come from pandas, on the native chart slide.
> </details>

---
<div style="background:#D49A00;color:white;padding:14px 18px;border-radius:8px;border-left:10px solid #F5C242;font-family:Calibri,Arial,sans-serif"><span style="font-size:13px;letter-spacing:3px;color:#F5C242;font-weight:bold">STATION D</span><br><b style="font-size:24px">Charts & graphics</b><br><span style="font-size:13px">⏱ 20 min &nbsp;·&nbsp; 🧰 Gemini API + matplotlib &nbsp;·&nbsp; 🎯 Deliverable: <code>D_chart.png</code> + <code>D_chart_card.md</code> (alt text, critique)</span></div>

### 🎯 Goal
Let an LLM propose **what** to plot, while Python controls **how** it is drawn — then generate **alt text** and a **design critique**.

### 💡 Concept — ask for a *chart specification*, not for arbitrary code
Letting a model write free plotting code works, but every chart ends up looking different and may contain subtle errors. A **specification** (a small JSON object) is easy to validate and renders in your house style every time.

**The chart checklist** used in the verification step:

| # | Rule |
|---|---|
| 1 | Bars start at zero |
| 2 | Title states the finding, not just the topic |
| 3 | Categories sorted by value (unless there is a natural order) |
| 4 | Source and units are shown |
| 5 | Colour is not the only carrier of meaning |
| 6 | Alt text describes the message for screen-reader users |

First, see why rule 1 matters:

In [ ]:
d = kivu.sort_values("internet_hh_pct")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(d.region, d.internet_hh_pct, color=AFDB["brick"]); axes[0].set_xlim(20, 75)
axes[0].set_title("WRONG · axis starts at 20: Highlands almost vanishes", color=AFDB["brick"])
axes[1].barh(d.region, d.internet_hh_pct, color=AFDB["green"]); axes[1].set_xlim(0, 80)
axes[1].set_title("RIGHT · zero baseline: Capital is 3.3× Highlands", color=AFDB["deep"])
for a in axes: a.set_xlabel("Households with internet access (%)")
plt.tight_layout(); plt.show()

### ▶️ D1 · Ask for a chart specification

In [ ]:
COLUMNS = {c: str(t) for c, t in kivu.dtypes.items()}
D_PROMPT = f"""
ROLE: Data-visualisation specialist at a national statistics office.
CONTEXT: DataFrame columns and types: {COLUMNS}. Description: {DATA_NOTE}
Summary: {kivu.describe().round(1).to_dict()}
TASK: Propose ONE chart that shows the relationship between household internet access and mobile download speed across regions.
FORMAT: JSON only:
{{"chart_type": "bar"|"barh"|"scatter"|"line", "x": column, "y": column, "label": column or null,
  "title": finding-style title (max 80 chars), "subtitle": units and year, "x_label": str, "y_label": str,
  "sort_by": column or null}}
CHECK: Use only the column names listed above.
"""
spec = extract_json(ask(D_PROMPT, task="D_spec", station="D", json_mode=True, temperature=0.3))
spec

### ✅ D2 · Validate the spec, then render in house style

In [ ]:
def validate_spec(spec, df):
    errs = []
    if spec.get("chart_type") not in {"bar", "barh", "scatter", "line"}: errs.append(f"unsupported chart_type {spec.get('chart_type')}")
    for k in ("x", "y", "label", "sort_by"):
        v = spec.get(k)
        if v and v not in df.columns: errs.append(f"{k}='{v}' is not a column")
    if len(spec.get("title", "")) > 90: errs.append("title too long")
    return errs

def render_chart(spec, df, path):
    d = df.sort_values(spec["sort_by"]) if spec.get("sort_by") else df
    fig, ax = plt.subplots(figsize=(9, 5.2))
    t = spec["chart_type"]
    if t == "scatter":
        ax.scatter(d[spec["x"]], d[spec["y"]], s=d["population_k"] / 6, color=AFDB["green"], alpha=.8, edgecolor=AFDB["deep"])
        if spec.get("label"):
            for _, r in d.iterrows(): ax.annotate(r[spec["label"]], (r[spec["x"]], r[spec["y"]]), xytext=(6, 4), textcoords="offset points", fontsize=9, color=AFDB["slate"])
        ax.set_xlim(left=0); ax.set_ylim(bottom=0)
    elif t in ("bar", "barh"):
        f = ax.barh if t == "barh" else ax.bar
        bars = f(d[spec["x"]], d[spec["y"]], color=AFDB["green"])
        ax.bar_label(bars, fmt="%.1f", padding=3, color=AFDB["slate"], fontsize=9)
    else:
        ax.plot(d[spec["x"]], d[spec["y"]], marker="o", color=AFDB["green"])
    ax.set_title(spec["title"], loc="left", pad=22)
    ax.text(0, 1.02, spec.get("subtitle", ""), transform=ax.transAxes, color=AFDB["slate"], fontsize=10)
    ax.set_xlabel(spec.get("x_label", spec["x"])); ax.set_ylabel(spec.get("y_label", spec["y"]))
    fig.text(0.01, 0.01, "Source: synthetic Kivuland dataset (training example). Bubble size = population.", fontsize=8, style="italic", color=AFDB["slate"])
    plt.tight_layout(); fig.savefig(path, dpi=200, bbox_inches="tight"); plt.show()

errs = validate_spec(spec, kivu)
r = float("nan")
if errs: banner("bad", "Spec rejected: " + "; ".join(errs))
else:
    render_chart(spec, kivu, OUT / "D_chart.png")
    r = kivu[spec["x"]].corr(kivu[spec["y"]]) if spec["chart_type"] == "scatter" else float("nan")
    banner("ok", f"💾 saved outputs/D_chart.png · correlation computed in pandas: r = {r:.2f}")

### ▶️ D3 · Alt text and critique — sending the *image* to Gemini

In [ ]:
assert (OUT / "D_chart.png").exists(), "Run D2 successfully first."
img_bytes = (OUT / "D_chart.png").read_bytes()
D3_PROMPT = f"""
ROLE: Accessibility and data-visualisation reviewer.
CONTEXT: The attached chart was produced from this spec: {json.dumps(spec)}. Verified correlation r = {r:.2f}.
TASK: (1) Write alt text of 40–60 words that states the main message.
(2) Review the chart against this checklist and give PASS/IMPROVE for each item with a short reason:
zero baseline; finding-style title; sensible ordering; source and units shown; not relying on colour alone.
FORMAT: Markdown with headings "Alt text" and "Checklist review" (a table).
CHECK: Only describe what is visible in the image or stated above.
"""
d_card = ask(D3_PROMPT, task="D_critique", station="D", images=[img_bytes], temperature=0.2)
show(d_card)
(OUT / "D_chart_card.md").write_text(f"# Chart card\n\n![chart](D_chart.png)\n\n## Specification\n```json\n{json.dumps(spec, indent=2)}\n```\n\n{d_card}\n")
print("💾 saved outputs/D_chart_card.md")

### ✍️ Your turn (D)
Change the TASK in `D_PROMPT` to *"show which regions saw the largest fall in inflation"*. Does the model pick a sensible chart type? Does the validator catch any mistakes?

> 🧠 **Check yourself.** Why is `r` computed in pandas rather than asked from the model?
> <details><summary>Show answer</summary>
> A correlation is a computation. Models approximate arithmetic from patterns and can be wrong with full confidence. Compute in code; let the model describe.
> </details>

---
<div style="background:#C4621D;color:white;padding:14px 18px;border-radius:8px;border-left:10px solid #F5C242;font-family:Calibri,Arial,sans-serif"><span style="font-size:13px;letter-spacing:3px;color:#F5C242;font-weight:bold">STATION E</span><br><b style="font-size:24px">Visual identity & logo concepts</b><br><span style="font-size:13px">⏱ 20 min &nbsp;·&nbsp; 🧰 Groq / Gemini → SVG &nbsp;·&nbsp; 🎯 Deliverable: <code>E_logo_1..3.svg</code> + <code>E_palette.png</code></span></div>

### 🎯 Goal
Produce **three logo concepts** and a **colour palette** for a fictional unit — the *Kivuland Data Innovation Lab* — and check the palette for **accessibility**.

### 💡 Concept — a logo can be *text*
**SVG** (Scalable Vector Graphics) is an XML text format. A text model can therefore draw simple, editable, infinitely scalable logos for free. SVG files open in PowerPoint, Word, Inkscape and every browser.

**Accessibility:** the Web Content Accessibility Guidelines (WCAG 2.2) require a **contrast ratio of at least 4.5 : 1** for normal text. We compute it — we do not trust the model's claim.

> ⚠️ **Intellectual property and approval.** AI concepts are a *starting point for a designer*, never a final logo. Do not reproduce national emblems, flags or other organisations' marks, and follow your office's approval process. For photographic mood boards, the free **Gemini app** (gemini.google.com) can generate images — same rules apply.

### ▶️ E1 · Brief → three SVG concepts

In [ ]:
E_BRIEF = """Unit: Kivuland Data Innovation Lab (fictional), part of the national statistics office.
Personality: trustworthy, modern, African, open. Themes: data points, connectivity, growth.
Constraints: flat design, 2–3 colours, legible at 64 px, no flags or national emblems, no text other than 'KDIL'."""
E_PROMPT = f"""
ROLE: Senior brand designer for public institutions.
CONTEXT: {E_BRIEF}
TASK: Create three distinct logo concepts as inline SVG.
FORMAT: JSON only: {{"concepts": [{{"name": str, "rationale": str (max 30 words), "svg": str}}]}}
Each svg: viewBox="0 0 200 200", width="200" height="200", only basic shapes (circle, rect, path, line, polygon, text),
no <script>, no external links, no embedded images.
CHECK: Make the three concepts visually different from each other.
"""
e_raw = ask(E_PROMPT, task="E_logos", station="E", provider="groq", json_mode=True, temperature=0.8)
concepts = extract_json(e_raw)["concepts"]
print(f"{len(concepts)} concepts received")

### ✅ E2 · Sanitise the SVG (security), display, save

In [ ]:
import xml.etree.ElementTree as ET
ALLOWED = {"svg", "g", "circle", "rect", "path", "line", "polygon", "polyline", "ellipse", "text", "tspan", "defs", "lineargradient", "stop", "title"}

def sanitize_svg(svg):
    """Reject anything that could run code or fetch external content."""
    issues = []
    if re.search(r"<script|javascript:|<foreignObject|<image|xlink:href\s*=\s*['\"]http|href\s*=\s*['\"]http", svg, re.I):
        issues.append("forbidden element or external link")
    try:
        root = ET.fromstring(svg)
        for el in root.iter():
            tag = el.tag.split("}")[-1].lower()
            if tag not in ALLOWED: issues.append(f"tag <{tag}> not allowed")
            if any(a.lower().startswith("on") for a in el.attrib): issues.append(f"event handler on <{tag}>")
    except ET.ParseError as e:
        issues.append(f"invalid XML: {e}")
    return issues

cards = []
for i, c in enumerate(concepts, 1):
    issues = sanitize_svg(c["svg"])
    if issues:
        cards.append(f'<div style="width:230px;padding:10px;border:1px solid {AFDB["brick"]};border-radius:8px">⛔ <b>{c["name"]}</b><br>{"; ".join(set(issues))}</div>')
        continue
    (OUT / f"E_logo_{i}.svg").write_text(c["svg"])
    cards.append(f'<div style="width:230px;padding:10px;border:1px solid {AFDB["sage"]};border-radius:8px;background:white;color:{AFDB["ink"]}">'
                 f'{c["svg"]}<br><b>{i}. {c["name"]}</b><br><span style="font-size:12px;color:{AFDB["slate"]}">{c["rationale"]}</span></div>')
display(HTML('<div style="display:flex;gap:14px;flex-wrap:wrap">' + "".join(cards) + "</div>"))
print("💾 saved the safe concepts as outputs/E_logo_*.svg")

### ▶️ E3 · Palette with a *computed* accessibility check

In [ ]:
P_PROMPT = f"""
ROLE: Brand designer. CONTEXT: {E_BRIEF}
TASK: Propose a 5-colour palette (primary, secondary, accent, dark text, light background).
FORMAT: JSON only: {{"palette": [{{"role": str, "name": str, "hex": "#RRGGBB"}}]}}
CHECK: Dark text must be readable on the light background and on white.
"""
palette = extract_json(ask(P_PROMPT, task="E_palette", station="E", provider="groq", json_mode=True))["palette"]

def luminance(hex_):
    rgb = [int(hex_.lstrip("#")[i:i+2], 16) / 255 for i in (0, 2, 4)]
    lin = [c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4 for c in rgb]
    return 0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2]
def contrast(a, b):
    la, lb = sorted([luminance(a), luminance(b)], reverse=True)
    return (la + 0.05) / (lb + 0.05)

rows = []
for p in palette:
    cw, cb = contrast(p["hex"], "#FFFFFF"), contrast(p["hex"], "#231F20")
    rows.append({**p, "vs white": f"{cw:.1f}:1 " + ("✅" if cw >= 4.5 else "❌"), "vs near-black": f"{cb:.1f}:1 " + ("✅" if cb >= 4.5 else "❌"),
                 "use for text on": "white" if cw >= 4.5 else ("dark" if cb >= 4.5 else "large text only")})
pal_df = pd.DataFrame(rows); display(pal_df)

fig, ax = plt.subplots(figsize=(10, 2.2))
for i, p in enumerate(palette):
    ax.add_patch(plt.Rectangle((i, 0.35), 0.92, 0.65, color=p["hex"]))
    txt = "#FFFFFF" if contrast(p["hex"], "#FFFFFF") >= contrast(p["hex"], "#231F20") else "#231F20"
    ax.text(i + .46, .68, "Aa", ha="center", va="center", fontsize=20, color=txt, weight="bold")
    ax.text(i + .46, .2, f'{p["role"]}\n{p["hex"]}', ha="center", va="center", fontsize=9, color=AFDB["slate"])
ax.set_xlim(0, len(palette)); ax.set_ylim(0, 1); ax.axis("off")
fig.savefig(OUT / "E_palette.png", dpi=200, bbox_inches="tight"); plt.show()
print("💾 saved outputs/E_palette.png")

### ✍️ Your turn (E)
Edit `E_BRIEF` for a real product of your office (a new dashboard, a census campaign, an open-data portal) and rerun E1–E3.

> 🧠 **Check yourself.** Why do we parse the SVG before displaying it?
> <details><summary>Show answer</summary>
> SVG can contain scripts and external links. Anything generated by a model — or pasted from the internet — is untrusted input. Sanitising it before display or publication is standard security hygiene.
> </details>

---
<div style="background:#0E7C86;color:white;padding:14px 18px;border-radius:8px;border-left:10px solid #F5C242;font-family:Calibri,Arial,sans-serif"><span style="font-size:13px;letter-spacing:3px;color:#F5C242;font-weight:bold">STATION F</span><br><b style="font-size:24px">Document analysis & audio briefings</b><br><span style="font-size:13px">⏱ 20 min &nbsp;·&nbsp; 🧰 NotebookLM (web) + Gemini + gTTS &nbsp;·&nbsp; 🎯 Deliverable: <code>F_grounded_answer.md</code> + <code>F_briefing.mp3</code></span></div>

### 🎯 Goal
Ask questions of a **set of documents** and get answers **with citations**, then turn the result into a short **audio briefing** for busy users.

### 💡 Concept — *grounding*
A grounded assistant answers **only from the sources you give it** and points to the passage it used. This is the idea behind the RAG laboratory of Day 1, and it is exactly what **NotebookLM** does in the browser.

| | Ungrounded chatbot | Grounded (NotebookLM / RAG) |
|---|---|---|
| Knowledge | whatever it learned in training | only your uploaded sources |
| Citations | often none, sometimes invented | links to the exact passage |
| Best for | brainstorming | document analysis, briefings |

### Part F-1 · NotebookLM in the browser (10 min)
1. Open **[notebooklm.google.com](https://notebooklm.google.com)** and sign in with a Google account → **New notebook**.
2. **Add sources** (public documents only): for example the HLG-MOS white paper *Large Language Models for Official Statistics* (UNECE, 2023), and your office's latest **published** statistical yearbook or methodology note (PDF or URL).
3. Ask three questions and **click each citation** to check the passage:
   * *"What are the three main risks of LLMs for statistical offices, according to these sources?"*
   * *"Which use cases have already been implemented by national offices?"*
   * *"What does the yearbook say about the coverage of [your survey]?"*
4. In the **Studio** panel, choose **Audio Overview** → *Customise* → e.g. *"Focus on quality and confidentiality; audience: senior managers; keep it short."* → Generate. Generation can take several minutes — continue with Part F-2 meanwhile.
5. Listen **against the sources**: note one thing the audio simplified or got wrong. Download the audio (⋮ menu).

> ⚠️ Label any AI-generated audio as such before sharing it. Features and quotas of free tools change; if Audio Overview is unavailable, Part F-2 produces an equivalent.

### Part F-2 · The same idea in code (10 min)
We build a mini-corpus of three **fictional** Kivuland documents, retrieve the relevant passages, ask Gemini for a **cited answer**, verify the citations, and produce an MP3.

In [ ]:
# @title F1 · A small fictional corpus, split into citable passages
CORPUS = [
 ("CPI-1", "CPI methodology note", "The Kivuland Consumer Price Index measures the change over time in the prices of goods and services bought by households. Prices are collected monthly in all eight regions, with about 4,200 price quotes per month."),
 ("CPI-2", "CPI methodology note", "Expenditure weights come from the most recent household budget survey and are updated every five years. Regional indices are aggregated to the national index using regional population as weights."),
 ("CPI-3", "CPI methodology note", "Rural outlets are under-represented in the Highlands and Lakes regions because of access constraints. Users should interpret regional differences for these two regions with caution."),
 ("NET-1", "Connectivity bulletin (experimental)", "The connectivity statistics are experimental. They are derived from crowdsourced speed tests published as open data under a CC BY-NC-SA 4.0 licence, aggregated to regions and weighted by gridded population."),
 ("NET-2", "Connectivity bulletin (experimental)", "Crowdsourced tests are not a probability sample: they over-represent urban users and people who own smartphones. Speeds therefore describe tested connections, not all households."),
 ("NET-3", "Connectivity bulletin (experimental)", "Regions with few tests per quarter are flagged. Results for flagged regions should not be used to rank regions."),
 ("REV-1", "Release and revision policy", "Official statistics are released on pre-announced dates. Revisions are made only to correct errors or to incorporate new weights, and they are announced in advance with an explanatory note."),
 ("REV-2", "Release and revision policy", "Experimental statistics are clearly labelled and may be revised or discontinued without the usual notice period."),
]
corpus = pd.DataFrame(CORPUS, columns=["id", "document", "text"])

def tokenize(t): return re.findall(r"[a-z]+", t.lower())
STOP = set("the of and to in a are is for by as be with from on or that this these their not all about per".split())
docs_tokens = [set(tokenize(t)) - STOP for t in corpus.text]
idf = {w: math.log(len(docs_tokens) / sum(w in d for d in docs_tokens)) + 1 for d in docs_tokens for w in d}

def retrieve(question, k=4):
    q = set(tokenize(question)) - STOP
    scores = [sum(idf.get(w, 0) for w in q & d) for d in docs_tokens]
    out = corpus.assign(score=np.round(scores, 2)).sort_values("score", ascending=False)
    return out[out.score > 0].head(k)

QUESTION = "How are regional figures weighted, and what limitations should users keep in mind for regional comparisons?"
hits = retrieve(QUESTION, k=5)
display(hits[["id", "document", "score", "text"]])

In [ ]:
# @title F2 · Grounded answer with citations — and a citation check
context = "\n".join(f"[{r.id}] ({r.document}) {r.text}" for r in hits.itertuples())
F_PROMPT = f"""
ROLE: Information officer answering a user query for a national statistics office.
CONTEXT — passages you may use (and nothing else):
{context}
TASK: Answer the question: "{QUESTION}"
FORMAT: 4–6 sentences in Markdown. End EVERY sentence with its source id in square brackets, e.g. [CPI-2].
CHECK: If the passages do not answer part of the question, say so explicitly. Do not use outside knowledge.
"""
f_answer = ask(F_PROMPT, task="F_answer", station="F", temperature=0.1)
show(f_answer)

cited = re.findall(r"\[([A-Z]+-\d+)\]", f_answer)
valid, retrieved_ids = set(corpus.id), set(hits.id)
sentences = [s for s in re.split(r"(?<=[.!?\]])\s+", f_answer.strip()) if len(s) > 20]
uncited = [s for s in sentences if not re.search(r"\[[A-Z]+-\d+\]", s)]
report = pd.DataFrame([{"citation": c, "exists": "✅" if c in valid else "❌ invented",
                        "was retrieved": "✅" if c in retrieved_ids else "⚠️ not in context"} for c in dict.fromkeys(cited)])
display(report)
coverage = 1 - len(uncited) / max(len(sentences), 1)
banner("ok" if coverage == 1 and (report["exists"] == "✅").all() else "warn",
       f"Citation coverage: {coverage:.0%} of sentences cite a source. Invented ids: {sum(report['exists'] != '✅')}. "
       "Now open each cited passage in the table above and confirm it supports the sentence.")
(OUT / "F_grounded_answer.md").write_text(f"# Grounded answer\n\n**Question:** {QUESTION}\n\n{f_answer}\n\n## Citation check\n{report.to_markdown(index=False)}\n\n## Sources\n{context}\n")
print("💾 saved outputs/F_grounded_answer.md")

In [ ]:
# @title F3 · Two-voice audio briefing (script by Gemini, voices by free gTTS)
F3_PROMPT = f"""
ROLE: Scriptwriter for a 90-second internal audio briefing of a statistics office.
CONTEXT: Use only this verified answer: {f_answer}
TASK: Write a dialogue between two hosts, Amina and Kofi, for senior managers.
FORMAT: JSON only: {{"lines": [{{"speaker": "Amina"|"Kofi", "text": str}}]}} — 6 to 8 lines, max 30 words each.
The first line must say that the briefing was generated with AI for training purposes.
CHECK: No figures and no claims beyond the verified answer.
"""
script = extract_json(ask(F3_PROMPT, task="F_script", station="F", json_mode=True, temperature=0.6))["lines"]
show("\n".join(f"**{l['speaker']}:** {l['text']}" for l in script))

try:
    from gtts import gTTS
    VOICES = {"Amina": "co.za", "Kofi": "com.ng"}   # English with different regional accents
    audio = io.BytesIO()
    for l in script:
        buf = io.BytesIO()
        gTTS(l["text"], lang="en", tld=VOICES.get(l["speaker"], "com")).write_to_fp(buf)
        audio.write(buf.getvalue())
    (OUT / "F_briefing.mp3").write_bytes(audio.getvalue())
    display(Audio(str(OUT / "F_briefing.mp3")))
    banner("ok", "💾 saved outputs/F_briefing.mp3 — listen and compare it with the cited answer.")
except Exception as e:
    (OUT / "F_briefing_script.md").write_text("\n\n".join(f"**{l['speaker']}:** {l['text']}" for l in script))
    banner("warn", f"Text-to-speech unavailable here ({type(e).__name__}). The script was saved to outputs/F_briefing_script.md — run this cell in Colab for the MP3.")

> 🧠 **Check yourself.** The answer contains a citation that *exists* but was *not retrieved*. What does that tell you?
> <details><summary>Show answer</summary>
> The model used an id it could not have read in this prompt — a red flag for hallucinated grounding. Existence of a citation is not enough; it must point to a passage the model actually received, and that passage must support the sentence.
> </details>

---
<div style="background:#00704A;color:white;padding:14px 18px;border-radius:8px;border-left:10px solid #F5C242;font-family:Calibri,Arial,sans-serif"><span style="font-size:13px;letter-spacing:3px;color:#F5C242;font-weight:bold">STATION G</span><br><b style="font-size:24px">Multimodal work with Gemini</b><br><span style="font-size:13px">⏱ 20 min &nbsp;·&nbsp; 🧰 Gemini API (vision, free tier) &nbsp;·&nbsp; 🎯 Deliverable: <code>G_extraction_report.md</code> with accuracy scores</span></div>

### 🎯 Goal
Use a **multimodal** model to read a **chart image** and a **scanned form**, and — above all — **measure** how accurate the extraction is.

### 💡 Concept — *multimodal* = several kinds of input
Gemini accepts text, images, PDFs and audio in the same request. For statistical offices this opens use cases such as digitising legacy paper tables, reading charts from old publications, or pre-filling data entry from scanned questionnaires.

The professional habit: **never trust an extraction you have not scored.** We create images whose true values we know, so we can compute the error — a miniature version of the *evaluation set* from the optimisation session.

```
known values ──► image ──► Gemini ──► JSON ──► compare with known values ──► accuracy
```

In [ ]:
# @title G1 · Create a chart image whose true values we know
truth_chart = dict(zip(kivu.region, kivu.dl_speed_mbps))
fig, ax = plt.subplots(figsize=(8, 4.2))
d = kivu.sort_values("dl_speed_mbps", ascending=False)
bars = ax.bar(d.region, d.dl_speed_mbps, color=RAMP[4])
ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=10)
ax.set_title("Median mobile download speed by region (Mbps)", loc="left"); ax.set_ylim(0, 30)
plt.xticks(rotation=20); plt.tight_layout()
chart_png = io.BytesIO(); fig.savefig(chart_png, format="png", dpi=110); plt.show()
chart_bytes = chart_png.getvalue()

In [ ]:
# @title G2 · Ask Gemini to read the chart, then score it
G_PROMPT = """
ROLE: Data-entry specialist digitising charts from old publications.
TASK: Read the attached bar chart and return the value of every bar.
FORMAT: JSON only: {"values": {"<region>": number, ...}}
CHECK: If a value is not legible, use null. Do not guess.
"""
got = extract_json(ask(G_PROMPT, task="G_chart", station="G", images=[chart_bytes], json_mode=True, temperature=0))["values"]
score = pd.DataFrame({"true": pd.Series(truth_chart), "extracted": pd.Series(got, dtype="float")})
score["abs_error"] = (score.extracted - score.true).abs().round(2)
score["exact"] = np.where(score.abs_error < 0.05, "✅", "❌")
display(score)
chart_acc = (score.exact == "✅").mean()
banner("ok" if chart_acc == 1 else "warn", f"Exact values: {chart_acc:.0%} · mean absolute error: {score.abs_error.mean():.2f} Mbps · missing: {score.extracted.isna().sum()}")

In [ ]:
# @title G3 · Create a synthetic "scanned" household form (tilted, noisy)
from PIL import Image as PILImage, ImageDraw, ImageFont, ImageFilter
truth_form = {"household_id": "KV-0417-22", "region": "Lakes", "household_size": 6,
              "head_age": 43, "water_source": "Protected well", "internet_at_home": "No"}
def _font(size):
    for f in ("DejaVuSans.ttf", "LiberationSans-Regular.ttf", "Arial.ttf"):
        try: return ImageFont.truetype(f, size)
        except Exception: pass
    return ImageFont.load_default()
im = PILImage.new("L", (900, 560), 245); dr = ImageDraw.Draw(im)
dr.text((40, 30), "HOUSEHOLD LISTING FORM — KIVULAND 2026 (SYNTHETIC)", fill=20, font=_font(26))
dr.line((40, 75, 860, 75), fill=60, width=2)
labels = ["Household ID", "Region", "Household size", "Age of head", "Main water source", "Internet at home (Yes/No)"]
for i, (lab, val) in enumerate(zip(labels, truth_form.values())):
    y = 110 + i * 70
    dr.text((40, y), lab + ":", fill=40, font=_font(22))
    dr.rectangle((420, y - 8, 860, y + 38), outline=90, width=2)
    dr.text((435 + random.Random(i).randint(0, 12), y), str(val), fill=15, font=_font(27))
im = im.rotate(1.8, expand=False, fillcolor=245).filter(ImageFilter.GaussianBlur(0.9))
arr = np.array(im).astype(int) + np.random.default_rng(7).normal(0, 12, (560, 900)).astype(int)
im = PILImage.fromarray(np.clip(arr, 0, 255).astype("uint8"))
form_buf = io.BytesIO(); im.save(form_buf, format="PNG"); form_bytes = form_buf.getvalue()
display(IPImage(form_bytes, width=600))

In [ ]:
# @title G4 · Transcribe the form and score field by field
G4_PROMPT = f"""
ROLE: Careful data-entry clerk.
TASK: Transcribe the attached scanned form.
FORMAT: JSON only with exactly these keys: {list(truth_form)}. household_size and head_age as integers.
CHECK: If a field is unreadable, use null. Do not correct or normalise the values.
"""
got_form = extract_json(ask(G4_PROMPT, task="G_form", station="G", images=[form_bytes], json_mode=True, temperature=0))
form_score = pd.DataFrame([{"field": k, "true": v, "extracted": got_form.get(k),
                            "match": "✅" if str(got_form.get(k)).strip().lower() == str(v).lower() else "❌"} for k, v in truth_form.items()])
display(form_score)
form_acc = (form_score.match == "✅").mean()
banner("ok" if form_acc == 1 else "warn", f"Field accuracy: {form_acc:.0%}. A single wrong digit in an ID can break a merge — this is why human verification or double entry is still needed.")

(OUT / "G_extraction_report.md").write_text(
    f"# Multimodal extraction report\n\n## Chart reading\nExact values: {chart_acc:.0%} · MAE {score.abs_error.mean():.2f}\n\n{score.to_markdown()}\n\n"
    f"## Form transcription\nField accuracy: {form_acc:.0%}\n\n{form_score.to_markdown(index=False)}\n\n"
    "## Conclusion (to be completed by the participant)\nWould this be accurate enough for production in my office? Under which controls?\n")
print("💾 saved outputs/G_extraction_report.md")

### ✍️ Your turn (G)
Upload a **public** chart or table image from one of your office's past publications (Colab: Files panel → Upload), load it with `Path("my_image.png").read_bytes()` and adapt `G_PROMPT`. You will not have the "truth" automatically — type the correct values for 5 cells yourself and score them.

> 🧠 **Check yourself.** Accuracy was 100 % on the chart. Can you now automate chart digitisation?
> <details><summary>Show answer</summary>
> Not yet. One clean, labelled chart is not an evaluation set. You would need dozens of varied, realistic images (faded scans, no data labels, stacked bars), a measured error rate, and a decision on which errors are acceptable — that is level 4, "automate", on the delegation ladder.
> </details>

---
<div style="background:#B83B2E;color:white;padding:12px 18px;border-radius:8px;border-left:10px solid #F5C242"><b style="font-size:20px">✔ Wrap-up</b> &nbsp;·&nbsp; ⏱ 5 minutes &nbsp;·&nbsp; everyone</div>

### W1 · Export your prompt log and package your deliverables
The prompt log records every call you made — station, provider, model, latency, and the prompt itself. The best prompts from all participants will feed the **reference manual on key data skills** (STG17 Action Plan, activity 4.2.1).

In [ ]:
import zipfile
log = pd.DataFrame(PROMPT_LOG)
log.to_csv(OUT / "prompt_log.csv", index=False)
if len(log):
    display(log.groupby(["station", "provider", "mode"]).agg(calls=("task", "count"), mean_latency_s=("latency_s", "mean")).round(2))
with zipfile.ZipFile("outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.glob("*")): z.write(f, f.name)
files = pd.DataFrame([{"file": f.name, "kB": round(f.stat().st_size / 1024, 1)} for f in sorted(OUT.glob("*"))])
display(files)
try:
    from google.colab import files as colab_files
    colab_files.download("outputs.zip")
except Exception:
    print("📦 outputs.zip is ready in the working folder.")

### W2 · Verification note (fill in — this is part of your deliverable)
Double-click this cell and complete it.

| Station | Deliverable | What I checked | How | Result | Would I publish it? |
|---|---|---|---|---|---|
| _e.g. B_ | _B_commentary.md_ | _every figure_ | _automatic fact-check + read against table_ | _1 wrong figure corrected_ | _after methodologist review_ |
| | | | | | |
| | | | | | |

### W3 · Self-assessment (feeds the STG17 competencies framework, activity 4.1.1)
Rate yourself from 1 (not at all) to 5 (confidently) — before and after this lab.

| I can… | Before | After |
|---|---|---|
| write a prompt with Role · Context · Task · Format · Check | | |
| obtain and validate JSON output from an LLM | | |
| verify figures in AI-drafted text automatically | | |
| explain which data may and may not go into a free AI tool | | |
| measure the accuracy of an AI extraction | | |

### W4 · Exit ticket
*One use case I will try in my office within the next month, with the tool, the data (public/synthetic) and the check I will apply:*

> …

### Three things to remember
1. **One recipe, many jobs** — Role · Context · Task · Format · Check.
2. **Computers compute, language models narrate** — every figure comes from code.
3. **Public or synthetic data only on free tiers** — when in doubt, it is red.

### Further reading
* HLG-MOS (2023). *Large Language Models for Official Statistics* — white paper. UNECE.
* HLG-MOS (2025). *Generative AI for Official Statistics* — report. UNECE.
* UNECE (2019). *Generic Statistical Business Process Model (GSBPM) v5.1.*
* W3C (2023). *Web Content Accessibility Guidelines (WCAG) 2.2.*
* Tool documentation: Google AI Studio · Groq Console · NotebookLM Help · python-pptx.

<div style="background:#E8F5EF;border:1px solid #00A86A;border-radius:8px;padding:10px 14px;margin-top:10px;color:#231F20">
<b>STG17 · Emerging Issues, Emerging Practice.</b> Training material — synthetic data only. AfDB-inspired visual style; not an official AfDB brand product.
</div>